# Chapter 2-3 — From Logistic Regression to MLP, with Backpropagation

Runnable companion to `docs/02_mlp_backpropagation.md`. Two parts:

- **Part A** — Logistic regression cannot solve XOR; a 2-2-1 MLP with a
  non-linear activation can. We train it and plot the decision boundary.
- **Part B** — Backpropagation on the same network *by hand* in NumPy, then
  cross-checked against PyTorch autograd. After this verification, every
  later chapter trusts autograd by default.

This notebook is generated by `src/build_chapter_02_mlp_from_scratch.py`.

## Setup

In [1]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")             # headless: works under nbconvert --execute
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__)

torch 2.12.0+cu130


# Part A — Logistic regression → MLP

## The XOR data

Four 2-D points labeled by their XOR.

In [2]:
X_xor = torch.tensor([[0., 0.],
                      [0., 1.],
                      [1., 0.],
                      [1., 1.]])
y_xor = torch.tensor([[0.], [1.], [1.], [0.]])

for x, y in zip(X_xor, y_xor):
    print(f"x={x.tolist()}  y={int(y.item())}")

x=[0.0, 0.0]  y=0
x=[0.0, 1.0]  y=1
x=[1.0, 0.0]  y=1
x=[1.0, 1.0]  y=0


## Try logistic regression first

A linear-then-sigmoid model. We expect it to *fail* on XOR — no straight line
separates the four points.

In [3]:
class LogReg(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(2, 1)
    def forward(self, x):
        return self.fc(x)            # raw logit; BCEWithLogitsLoss handles sigmoid


def train_binary(model, X, y, lr=0.1, steps=2000):
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    for step in range(steps):
        opt.zero_grad()
        logits = model(X)
        loss = loss_fn(logits, y)
        loss.backward()
        opt.step()
    return loss.item()


def accuracy(model, X, y):
    with torch.no_grad():
        preds = (torch.sigmoid(model(X)) > 0.5).float()
        return (preds == y).float().mean().item()


lr_model = LogReg()
final_loss = train_binary(lr_model, X_xor, y_xor)
acc = accuracy(lr_model, X_xor, y_xor)
print(f"logistic regression on XOR: final loss={final_loss:.4f}, accuracy={acc:.2f}")
print("(expect ~50% — logistic regression cannot solve XOR)")

/home/bangbc/miniforge3/envs/aicourse/lib/python3.11/site-packages/torch/autograd/graph.py:882: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


logistic regression on XOR: final loss=0.6931, accuracy=0.25
(expect ~50% — logistic regression cannot solve XOR)


## A 2-2-1 MLP with ReLU solves XOR

One hidden layer, two units, ReLU. Same training recipe.

In [4]:
class MLP(nn.Module):
    def __init__(self, hidden=4):
        super().__init__()
        self.fc1 = nn.Linear(2, hidden)
        self.fc2 = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))


mlp = MLP(hidden=4)
final_loss = train_binary(mlp, X_xor, y_xor, lr=0.1, steps=4000)
acc = accuracy(mlp, X_xor, y_xor)
print(f"MLP on XOR: final loss={final_loss:.4f}, accuracy={acc:.2f}")

MLP on XOR: final loss=0.4777, accuracy=0.75


## Decision boundary

Evaluate both models on a fine 2-D grid and color by predicted class. The
linear model's boundary is a straight line; the MLP's boundary is piecewise
linear and bends around the four corners.

In [5]:
def plot_boundary(model, X, y, ax, title):
    grid_x, grid_y = torch.meshgrid(
        torch.linspace(-0.2, 1.2, 200),
        torch.linspace(-0.2, 1.2, 200),
        indexing="xy",
    )
    grid = torch.stack([grid_x.flatten(), grid_y.flatten()], dim=1)
    with torch.no_grad():
        probs = torch.sigmoid(model(grid)).reshape(grid_x.shape)
    ax.contourf(grid_x, grid_y, probs, levels=20, cmap="coolwarm", alpha=0.6)
    ax.scatter(X[:, 0], X[:, 1], c=y.flatten(), cmap="coolwarm",
               edgecolors="k", s=120)
    ax.set_title(title)
    ax.set_xlabel("x1"); ax.set_ylabel("x2")


fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_boundary(lr_model, X_xor, y_xor, axes[0], "Logistic regression (linear)")
plot_boundary(mlp,      X_xor, y_xor, axes[1], "MLP with ReLU (non-linear)")
plt.tight_layout()
plt.savefig("xor_boundaries.png", dpi=100)
plt.show()
print("Saved xor_boundaries.png")

Saved xor_boundaries.png


/tmp/ipykernel_168403/3133463789.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Part B — Backpropagation by hand

We pick a *single* example from XOR (`x = [1, 0]`, `y = 1`) and a 2-2-1
sigmoid-output MLP. We will:

1. Compute the forward pass by hand.
2. Apply the chain rule, by hand, to derive `dL/dW1, db1, dW2, db2`.
3. Re-implement the same gradients in NumPy.
4. Compute the *same* gradients via PyTorch autograd.
5. Assert they match to within floating-point noise.

After this once, you have permission to trust autograd forever after.

## Set up the example

Fixed weights so the numbers are reproducible. The activation is
`sigmoid → ReLU → sigmoid` (sigmoid output, ReLU hidden).

In [6]:
def sigmoid(z): return 1 / (1 + np.exp(-z))
def relu(z):    return np.maximum(0, z)


x = np.array([1.0, 0.0])
y = np.array([1.0])

W1 = np.array([[ 0.30, -0.40],
               [ 0.20,  0.10]], dtype=np.float64)
b1 = np.array([0.05, -0.02], dtype=np.float64)
W2 = np.array([[0.50, -0.30]], dtype=np.float64)
b2 = np.array([0.10], dtype=np.float64)

print("x:", x, "y:", y)
print("W1:\n", W1, "\nb1:", b1)
print("W2:\n", W2, "\nb2:", b2)

x: [1. 0.] y: [1.]
W1:
 [[ 0.3 -0.4]
 [ 0.2  0.1]] 
b1: [ 0.05 -0.02]
W2:
 [[ 0.5 -0.3]] 
b2: [0.1]


## Forward pass — NumPy

In [7]:
def forward(x, W1, b1, W2, b2):
    z1   = W1 @ x + b1
    h    = relu(z1)
    z2   = W2 @ h + b2
    yhat = sigmoid(z2)
    return z1, h, z2, yhat


z1, h, z2, yhat = forward(x, W1, b1, W2, b2)
print(f"z1 = {z1}")
print(f"h  = {h}")
print(f"z2 = {z2}")
print(f"yhat = {yhat}  (= sigmoid(z2))")

# BCE loss
loss_np = -(y * np.log(yhat) + (1 - y) * np.log(1 - yhat))
print(f"loss = {loss_np}")

z1 = [0.35 0.18]
h  = [0.35 0.18]
z2 = [0.221]
yhat = [0.55502622]  (= sigmoid(z2))
loss = [0.58873992]


## Backward pass — NumPy

Chain rule, by hand. For sigmoid + BCE, `dL/dz_2 = ŷ - y` is the famous
simplification. From there we propagate back layer by layer.

In [8]:
def backward(x, y, W1, b1, W2, b2):
    z1, h, z2, yhat = forward(x, W1, b1, W2, b2)

    dz2 = yhat - y                                  # (1,)
    dW2 = dz2[:, None] * h[None, :]                  # (1, 2)
    db2 = dz2.copy()                                 # (1,)

    dh  = W2.T @ dz2                                 # (2,)
    dz1 = dh * (z1 > 0).astype(np.float64)           # (2,)
    dW1 = dz1[:, None] * x[None, :]                  # (2, 2)
    db1 = dz1.copy()                                 # (2,)

    return dW1, db1, dW2, db2


dW1, db1, dW2, db2 = backward(x, y, W1, b1, W2, b2)
print("dW1:\n", dW1)
print("db1:",  db1)
print("dW2:\n", dW2)
print("db2:",  db2)

dW1:
 [[-0.22248689 -0.        ]
 [ 0.13349213  0.        ]]
db1: [-0.22248689  0.13349213]
dW2:
 [[-0.15574082 -0.08009528]]
db2: [-0.44497378]


## Backward pass — PyTorch autograd

Same example, same starting weights, but every tensor `requires_grad=True`.
PyTorch records the operations and `.backward()` computes the gradients for
us. We then compare to the NumPy result element-by-element.

In [9]:
xt  = torch.tensor(x,  dtype=torch.float64)
yt  = torch.tensor(y,  dtype=torch.float64)

W1t = torch.tensor(W1, dtype=torch.float64, requires_grad=True)
b1t = torch.tensor(b1, dtype=torch.float64, requires_grad=True)
W2t = torch.tensor(W2, dtype=torch.float64, requires_grad=True)
b2t = torch.tensor(b2, dtype=torch.float64, requires_grad=True)

z1_t   = W1t @ xt + b1t
h_t    = torch.relu(z1_t)
z2_t   = W2t @ h_t + b2t
yhat_t = torch.sigmoid(z2_t)
loss_t = F.binary_cross_entropy(yhat_t, yt)
loss_t.backward()

print("loss (torch) =", loss_t.item(), "vs numpy =", float(loss_np.item()))
print("W1.grad:\n", W1t.grad.numpy())
print("b1.grad:", b1t.grad.numpy())
print("W2.grad:\n", W2t.grad.numpy())
print("b2.grad:", b2t.grad.numpy())

loss (torch) = 0.5887399216808016 vs numpy = 0.5887399216808016
W1.grad:
 [[-0.22248689 -0.        ]
 [ 0.13349213  0.        ]]
b1.grad: [-0.22248689  0.13349213]
W2.grad:
 [[-0.15574082 -0.08009528]]
b2.grad: [-0.44497378]


## Cross-check

The NumPy gradients and the autograd gradients should match to within
floating-point noise (`~1e-10` in float64).

In [10]:
def assert_close(name, a, b, atol=1e-10):
    diff = np.abs(a - b).max()
    print(f"{name}: max abs diff = {diff:.2e}")
    assert diff < atol, f"{name} mismatch"


assert_close("dW1", dW1, W1t.grad.numpy())
assert_close("db1", db1, b1t.grad.numpy())
assert_close("dW2", dW2, W2t.grad.numpy())
assert_close("db2", db2, b2t.grad.numpy())
print("\nAll four hand-derived gradients match autograd.")

dW1: max abs diff = 2.78e-17
db1: max abs diff = 2.78e-17
dW2: max abs diff = 2.78e-17
db2: max abs diff = 5.55e-17

All four hand-derived gradients match autograd.


## Why we did this exactly once

From here, every later chapter writes:

```python
loss = criterion(model(x), y)
loss.backward()
optimizer.step()
```

…and trusts autograd. The point of this notebook was to *verify* that trust
on a small example — not to do it ever again.

## Next

- `docs/03_training_loop.md` — the canonical PyTorch training loop.
- `projects/project_01_mlp_mnist/` — apply the loop to an MLP on MNIST.